# Week 9 Post-Class Fun: The Modern CV Playground 🎡

In the live session you saw a whirlwind tour of what modern computer vision can do beyond
classification. This notebook lets you **actually run all three** — tonight, on your laptop,
on your own photos if you want.

| Part | Task | The question it answers | Model you'll run |
|---|---|---|---|
| 1 | **Object detection** | *What's in this image, and where?* | YOLO |
| 2 | **Segmentation** | *Which exact pixels belong to what?* | DeepLabV3 |
| 3 | **Zero-shot classification** | *Which of MY descriptions fits this image?* | CLIP |

**Three things to know before you start:**

1. **You will not train anything.** Every model here is *pretrained* — someone already spent
   the GPU-months. You just download the finished weights and run them. (That's the entire
   moral of Week 9: start from pretrained.)
2. **Everything runs on a plain CPU** in a few seconds per image. No GPU needed.
3. Parts 1 and 2 use the **same street photo** — same image, two different questions. Watch
   how differently the two models answer.

⏱️ About 15 minutes. The models download themselves the first time you run each part
(~6 MB, ~160 MB, ~600 MB) — after that they're cached on your machine.

## Setup: one new package

You already have `torch`, `torchvision`, `transformers`, and `matplotlib` from the course
environment. The only **new** package is `ultralytics` (the YOLO library).

**To install it:** uncomment the `%pip` line below and run the cell. (`%pip` installs into
the exact Python environment this notebook is using — safer than typing `pip` in a random
terminal.) After it finishes, restart the kernel (menu: *Kernel → Restart*) and run the
notebook from the top with the line commented out again.

In [ ]:
%pip install ultralytics    # ← uncomment, run once, restart kernel, re-comment

import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

print('torch', torch.__version__, '| running on CPU — that is plenty for inference')

In [ ]:
# Download the two photos we'll play with (only happens once).
# street.jpg: a street scene with people and a bus — the SAME photo from the live-session tour.
# cats.jpg:   two cats on a couch, a classic computer-vision test image.
IMAGES = {
    'street.jpg': 'https://ultralytics.com/images/bus.jpg',
    'cats.jpg': 'http://images.cocodataset.org/val2017/000000039769.jpg',
}
for name, url in IMAGES.items():
    if not Path(name).exists():
        print('downloading', name, '...')
        urllib.request.urlretrieve(url, name)

street = Image.open('street.jpg').convert('RGB')
cats = Image.open('cats.jpg').convert('RGB')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(street); axes[0].set_title('street.jpg — for Parts 1 & 2'); axes[0].axis('off')
axes[1].imshow(cats); axes[1].set_title('cats.jpg — for Part 3'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Part 1: Object Detection with YOLO — *what AND where*

Your flower model answers one question about the **whole image**: "what is this?" One image
in, one label out. But the street photo has several people *and* a bus. **Object detection**
finds every object, draws a rectangle around each one (a **bounding box**), and labels it
with a class and a **confidence** — a score between 0 and 1 for how sure the model is.

### ELI5: how does one model find everything at once?

The obvious way would be to slide a small classifier across the image like a magnifying
glass — check this patch, slide over, check again — thousands of times, at many sizes. Old
detectors really worked that way. It was painfully slow.

**YOLO's insight: the network already scanned the image.** A CNN works by sliding little
filters across the whole picture — that *is* what a convolution does. By its last layer, the
network holds a coarse **grid of summaries** — think of the image divided into, say, 13×13
squares, where each square holds a rich description of what's around it.

Now the trick. Your flower model **averaged that grid away** into one summary, then made one
prediction. YOLO simply *doesn't average*. It keeps the grid, and at **every square** it
predicts a small bundle of numbers: *is there an object centered in my square? what class?
and where exactly is its box?* All squares answer **simultaneously, in one pass** — that's
the name: **Y**ou **O**nly **L**ook **O**nce.

Picture a stadium field divided into squares with a spotter standing on each one, all
watching at the same time. Whistle blows — every spotter raises a card at once: *"a dog is
centered on my square, and its body extends this far beyond it"* or *"nothing centered
here."* Two small rules keep it tidy: each object is *owned* by the one square its center
falls in (so forty spotters don't all claim the same dog), and a final cleanup step keeps
the most confident card and discards near-duplicate overlapping boxes.

And the part you already understand: underneath, YOLO is a **pretrained CNN backbone** — the
same kind of feature extractor you used on flowers — with a "boxes-and-labels head" bolted
on top. Same pattern as Week 9, fancier head.

In [ ]:
from ultralytics import YOLO

# 'n' is for nano — the smallest YOLO (~6 MB). Downloads itself the first time.
model = YOLO('yolov8n.pt')

results = model('street.jpg', verbose=False)
result = results[0]

# result.plot() returns the image with boxes drawn on it (in BGR order, so we flip to RGB)
plt.figure(figsize=(7, 8))
plt.imshow(result.plot()[..., ::-1])
plt.axis('off')
plt.title('YOLO: every object, found in ONE pass')
plt.show()

print('What YOLO found:')
for box in result.boxes:
    cls_name = result.names[int(box.cls)]
    conf = float(box.conf)
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    print(f'  {cls_name:<10} confidence {conf:.2f}   box corners ({x1},{y1}) → ({x2},{y2})')

**Things to notice:**

- It found the people *and* the bus in a single pass — and it took well under a second.
  That speed is why YOLO runs on live video: self-driving perception, sports tracking, drones.
- Each detection is exactly what the ELI5 promised: a class, a confidence, and four numbers
  (the box corners).
- This is the *nano* model. Bigger versions (`yolov8s.pt`, `yolov8m.pt`, …) are more accurate
  and slower — the eternal trade-off.

**🎮 Your turn:** drop any photo into this folder and run
`model('your_photo.jpg', verbose=False)` — pets, your desk, a crowd. YOLO knows 80 everyday
object classes (people, cars, dogs, cups, laptops…).

## Part 2: Segmentation with DeepLabV3 — *which exact pixels*

A bounding box is a rough answer — a rectangle around the bus also contains road, sky, and
half a person. **Semantic segmentation** answers precisely: it assigns a class to **every
single pixel**. Think of it as a coloring book: every bus pixel gets colored "bus," every
person pixel "person," everything else "background."

### ELI5: same trick as YOLO, pushed further

If YOLO's move was "keep the 13×13 grid and answer at every square," segmentation's move is
"keep the grid **and blow it back up to full size**, then answer at every **pixel**." The
network shrinks the image down into that grid of rich summaries (the same pretrained CNN
backbone again — ResNet-50, trained on ImageNet), then an upsampling head stretches the
answers back out until there's a class score for all ~400,000 pixels.

One vocabulary pair worth knowing: what we run here is **semantic** segmentation — all
people share one color, as a category. **Instance** segmentation (Mask R-CNN) goes further
and keeps person #1 separate from person #2. Same idea, more bookkeeping.

Where you've met this: medical imaging ("which pixels are the tumor"), background blur on
video calls, satellite mapping ("which pixels are flooded").

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

weights = DeepLabV3_ResNet50_Weights.DEFAULT   # pretrained — downloads once (~160 MB)
seg_model = deeplabv3_resnet50(weights=weights).eval()
preprocess = weights.transforms()               # the exact normalization this model expects
categories = weights.meta['categories']

with torch.no_grad():                           # inference only — no training, no gradients
    batch = preprocess(street).unsqueeze(0)     # add a batch dimension: [1, 3, H, W]
    out = seg_model(batch)['out'][0]            # class scores per pixel: [21, H, W]
mask = out.argmax(0).numpy()                    # per pixel, keep the highest-scoring class

found = [categories[i] for i in np.unique(mask) if categories[i] != '__background__']
print('Classes found in the photo:', ', '.join(found))

# Color the mask and blend it over the photo
rng = np.random.RandomState(9)
palette = rng.randint(60, 255, size=(len(categories), 3), dtype=np.uint8)
palette[0] = 0                                   # background stays black
color_mask = palette[mask]
color_img = Image.fromarray(color_mask).resize(street.size)
overlay = Image.blend(street, color_img, alpha=0.55)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, im, title in zip(axes, [street, color_img, overlay],
                         ['original', 'per-pixel classes', 'overlay']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

**Things to notice:**

- Compare with YOLO's rectangles: the segmentation follows the actual *outline* of the bus
  and the people — every pixel got its own answer.
- Look closely at the edges (around arms, between person and bus). Boundaries are where
  segmentation models sweat — deciding a pixel on the border is genuinely ambiguous.
- The backbone underneath is ResNet-50 pretrained on ImageNet. Third appearance of the same
  Week 9 pattern: pretrained backbone + task-specific head.

## Part 3: CLIP — classification where YOU type the classes

Everything so far has a fixed menu. YOLO knows its 80 objects; the segmenter knows 21
classes; your flower model knows exactly 5 flowers — show it a cat and it will confidently
pick a flower, because those 5 are all it can say. The menu is baked in at training time.

**CLIP throws away the menu.** You hand it an image *and a list of descriptions you just
made up*, and it tells you which description fits best. No training, no fixed classes. This
is called **zero-shot** classification — "zero" because the model gets zero examples of your
specific labels before making the call.

### ELI5: two translators, one shared map

CLIP is two models trained side by side on **400 million image–caption pairs** from the
internet: one reads images, one reads text, and both translate what they read into a point
on the same giant "map of meanings." (The coordinates of a point are called an
**embedding** — just a long list of numbers.) Training pushed each image and its true
caption to land at the *same spot* on the map, and mismatched pairs far apart.

After 400 million examples, classification stops being a special skill and becomes
**geography**: translate the image to the map, translate each of your candidate sentences to
the map, and see whose point lands **closest** to the image's point. Closest sentence wins.
The percentages below are just "closeness," rescaled to add up to 1.

In [ ]:
from transformers import CLIPModel, CLIPProcessor, logging as hf_logging

hf_logging.set_verbosity_error()   # hide harmless "load report" chatter from the library

clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')   # downloads once (~600 MB)
processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

def zero_shot(image, labels):
    """Score each text label against the image; print a little leaderboard."""
    inputs = processor(text=labels, images=image, return_tensors='pt', padding=True)
    with torch.no_grad():
        probs = clip(**inputs).logits_per_image.softmax(dim=1)[0]
    for p, label in sorted(zip(probs.tolist(), labels), reverse=True):
        print(f'  {p:6.1%}  {label}')
    print()

print('cats.jpg — classes I just made up:')
zero_shot(cats, [
    'two cats being extremely lazy on a couch',
    'a dog heroically catching a frisbee',
    'a bowl of petunias',
    'a rocket launching into space',
])

print('street.jpg — same model, completely different labels:')
zero_shot(street, [
    'a busy city street with a bus',
    'a quiet beach at sunset',
    'the inside of a refrigerator',
])

**🎮 Your turn — this is the fun one.** Edit the label lists and re-run. Ideas:

- Load your own pet's photo and ask: `['a good boy', 'a very good boy', 'the goodest boy']`
- Try labels the model must *reason* about: `['a photo taken indoors', 'a photo taken outdoors']`
- Try to fool it — how close do two labels have to be before it gets confused?

Notice what just happened: you built a brand-new classifier by *typing sentences*. No
dataset, no training loop, no epochs. This idea — pairing images with language — is the
bridge to Week 11, where models generate images *from* text using the very same shared map.

## The one pattern to take away

| Model | Output | Head on top | Backbone underneath |
|---|---|---|---|
| Your flower model | 1 label per image | GlobalAveragePooling + Dense | MobileNetV2, ImageNet-pretrained |
| YOLO | boxes + labels | box-and-class head at every grid square | pretrained CNN |
| DeepLabV3 | a label per pixel | upsampling head | ResNet-50, ImageNet-pretrained |
| CLIP | "which sentence fits" | shared image/text embedding map | pretrained vision + text encoders |

Four different superpowers, one identical recipe: **a pretrained backbone that already knows
how to see, plus a head shaped for the question you're asking.** You learned that recipe on
Tuesday with 500 flower photos. Everything you ran tonight is that same recipe scaled up.

And notice one more time: **you trained nothing today.** The era of "download the finished
model and build on it" is the era you're entering — next week we do it for text.